In [ ]:
import decoder
from qdk import qsharp, code
from qdk.widgets import Histogram
from qdk.simulation import NoiseConfig


# Initialize Q# with the project and adaptive profile configured
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive, project_root=".")

In [ ]:
%%qsharp

import C12;
import Std.Arrays.*;
import Std.Diagnostics.Fact;

/// Prepare a C12 block in the |00> state, with the option to prepare a given number of redundant candidates in parallel.
/// Returns true if preparation was successful, false if all candidates failed.
operation ParallelPrepZZ(block : Qubit[], num_redundant : Int) : Bool {
    Fact(num_redundant >= 0, "Number of redundant candidates must be non-negative.");
    let block_size = Length(block);

    // Allocate `num_redundant` additional blocks for preparation.
    use qs = Qubit[block_size * num_redundant];
    let prep_blocks = Chunks(block_size, block + qs);

    // Prepare all blocks in parallel, collecting the resulting preselect measurements.
    mutable preselect = [];
    parallel for block in prep_blocks {
        use ancillas = Qubit[4];
        preselect += [C12.PrepareZZ(block, ancillas)];
    }

    // Determine which block is the first successful preparation, swapping it into the main block if necessary.
    mutable failed = true;
    for i in 0..Length(preselect) - 1 {
        mutable is_all_zero = true;
        for r in preselect[i] {
            if IsLossResult(r) or r != Zero {
                is_all_zero = false;
                break;
            }
        }
        if is_all_zero {
            if i > 0 {
                for j in 0..block_size-1 {
                    SWAP(block[j], prep_blocks[i][j]);
                }
            }
            failed = false;
        }
    }

    // Reset the redundant blocks so they can be reused,
    // and return whether preparation failed.
    ResetAll(qs);
    failed
}

operation ParallelPrepXX(block : Qubit[], num_redundant : Int) : Bool {
    let failed = ParallelPrepZZ(block, num_redundant);
    ApplyToEach(H, block);
    Relabel(block[1..2], [block[2], block[1]]);
    Relabel(block[5..6], [block[6], block[5]]);
    Relabel(block[9..10], [block[10], block[9]]);
    failed
}

operation Experiment(num_redundant : Int) : (Bool, (Result[], Result[])[], Result[])[] {
    use block = Qubit[12];
    let prep_results = ParallelPrepZZ(block, num_redundant);
    let final = MResetEachZ(block);
    [(prep_results, [], final)]
}

In [ ]:
results = qsharp.run(code.Experiment, 1000, 0, type="clifford")
corrected_logical_results = decoder.decode_results2(results, "Z")

Histogram(map(str, corrected_logical_results))

In [ ]:
noise = NoiseConfig()

noise.h.set_depolarizing(0.00225)

noise.cx.ix = 0.0105 / 3
noise.cx.xi = 0.0105 / 3
noise.cx.xx = 0.0105 / 3

noise.cx.loss = 0.0027

num_redundant_preps = 0

results = qsharp.run(code.Experiment, 1000, num_redundant_preps, type="clifford", noise=noise)
corrected_logical_results = decoder.decode_results2(results, "Z")

Histogram(map(str, corrected_logical_results))


In [ ]:
%%qsharp
import Std.Diagnostics.Fact;
import Utils.TransversalCNOT;
import C12;

operation PerformTeleportExperiment(ec_repetitions : Int, num_redundant : Int) : (Bool, (Result[], Result[])[], Result[])[] {
    let block_size = 12;
    use logical_block = Qubit[block_size];

    // Prepare in the requested basis
    mutable preselect = ParallelPrepZZ(logical_block, num_redundant);

    mutable syndromes = [];

    for _ in 1..(ec_repetitions) {
        // Sequential teleport on..
        // Prepare Z, Teleport X
        use ancilla_block = Qubit[block_size];
        let prep_fail = ParallelPrepZZ(ancilla_block, num_redundant);
        preselect or= prep_fail;
        TransversalCNOT(logical_block, ancilla_block);
        ApplyToEach(H, logical_block);
        let syndrome_x = MResetEachZ(logical_block);

        // Prepare X, Teleport Z
        let prep_fail = ParallelPrepXX(logical_block, num_redundant);
        preselect or= prep_fail;
        TransversalCNOT(logical_block, ancilla_block);
        let syndrome_z = MResetEachZ(ancilla_block);
        set syndromes += [(syndrome_x, syndrome_z)];
    }

    // Final measurement
    let final = MResetEachZ(logical_block);

    [(preselect, syndromes, final)]
}

In [ ]:
ec_repetitions = 3
num_redundant_preps = 0
results = qsharp.run(code.PerformTeleportExperiment, 1000, ec_repetitions, num_redundant_preps, type="clifford", noise=noise)
corrected_logical_results = decoder.decode_results2(results, "Z")
print(f"Num Qubits: {qsharp.logical_counts(code.PerformTeleportExperiment, ec_repetitions, num_redundant_preps)['numQubits']}")
Histogram(map(str, corrected_logical_results))

In [ ]:
qir = qsharp.compile(code.PerformTeleportExperiment, ec_repetitions, num_redundant_preps)
print(qir)